# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a Croissant-formatted dataset using the `mlcroissant` library.

### Dataset Source
The FAIR² dataset is accessible via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print out dataset overview
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

> **Note**: All `mlcroissant` entities (record sets, fields, columns) are referenced by their unique `@id` as required.

In [ ]:
# List all record sets defined in the dataset
print("Record Sets (@id):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name']}")

# For demonstration, print fields and columns for each record set
record_sets_by_id = {rs['@id']: rs for rs in record_sets}
for rs_id, rs in record_sets_by_id.items():
    print(f"\nRecord Set: {rs['name']} (@id={rs_id})")
    if 'fields' in rs:
        print("  Fields (@id):")
        for field in rs['fields']:
            print(f"    - {field['@id']}: {field.get('name', field['@id'])}")
            if 'column' in field:
                print(f"       Column: {field['column']['@id']} ({field['column'].get('name','')})")
    else:
        print("  No fields defined.")

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. Use record set and field `@id`s as reviewed above.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
# Load data from each record set using its @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only add if data exists
        dataframes[record_set_id] = pd.DataFrame(records)

# Display available DataFrames and their sample columns
for rs_id, df in dataframes.items():
    print(f"\nRecord Set @id: {rs_id}")
    print("Columns:", df.columns.tolist())
    display(df.head())
    break  # Show only the first dataframe for brevity

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records by criteria, normalize fields, and group data by relevant fields using their `@id`.

In [ ]:
# Example: Select a numeric field from a record set (update these with the actual @id's shown above)
# Let's auto-detect the first numeric field in the first available dataframe
import numpy as np

if dataframes:
    # Take the first dataframe for demonstration
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")

    # Find first numeric field
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break

    if numeric_field_id is None:
        print("No numeric fields found for analysis.")
    else:
        threshold = df[numeric_field_id].quantile(0.8)  # Use upper 20% quantile for filtering
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try grouping by first non-numeric field
        group_field_id = None
        for col in df.columns:
            if not np.issubdtype(df[col].dtype, np.number):
                group_field_id = col
                break
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric field found for grouping.")
else:
    print("No record set with data available for EDA.")

## 5. Visualization
Visualize distributions and possible relationships in the dataset fields using their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8, 6))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR² dataset using the `mlcroissant` library, reviewed its record sets and field structure via their `@id`, extracted data for quick analysis, performed basic normalization and grouping, and visualized key numeric attributes. This approach demonstrates how Croissant metadata enables repeatable and schema-aware data exploration workflows for complex research datasets.

Further analysis can include model building, reporting bias review, or policy-relevant findings utilizing transparent field and record set references via their `@id`.